In [2]:
# imports

import os
import re
import math
import json
import random
import package.global_vars
from huggingface_hub import login
import matplotlib.pyplot as plt
import numpy as np
import pickle
from collections import Counter
from openai import OpenAI
from anthropic import Anthropic

In [3]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
from package.items import Item
from Tester import Tester

In [5]:
openai = OpenAI()
%matplotlib inline

In [14]:

with open('train_lite.pkl', 'rb') as file:
    train = pickle.load(file)

with open('test_lite.pkl', 'rb') as file:
    test = pickle.load(file)
fine_tune_train = train[:200]
fine_tune_validation = train[200:250]

In [16]:
def messages_for(item):
    system_message = "You estimate prices of items. Reply only with the price, no explanation"
    user_prompt = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"}
    ]

def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()
    
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)   


def get_price(s):
    s = s.replace('$','').replace(',','')
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0

In [ ]:
write_jsonl(fine_tune_train, "fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "fine_tune_validation.jsonl")

In [6]:
with open("fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [7]:
WANDB_ENTITY = "ananthakrishnan-a-s" 
WANDB_PROJECT = "gpt-pricer"
RUN_NAME = "gpt4o-mini-pricer-appliances" 
wandb_integration = {"type": "wandb", "wandb": {
    "project": WANDB_PROJECT,
    "entity": WANDB_ENTITY,
    "name": RUN_NAME,  
    "tags": ["gpt-4o-mini", "pricing","lite","appliances"]
    }
                    }
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    seed=42,
    hyperparameters={"n_epochs":1},
    integrations=[wandb_integration],
    suffix="pricer"
)

FineTuningJob(id='ftjob-qJjQ4lSBSlt9yNVjMziJDZ3M', created_at=1756524448, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-qKNBAjnqIPzQSWpbloplIxqp', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-FNyc5h3hrLjrE8GXt517P3', validation_file='file-YJLbNXNzvzLGz5BwoewzqZ', estimated_finish=None, integrations=[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity='ananthakrishnan-a-s', name=None, tags=None, run_id='ftjob-qJjQ4lSBSlt9yNVjMziJDZ3M'))], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1))), user_provi

In [8]:
openai.fine_tuning.jobs.list(limit=1)
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
job_id
openai.fine_tuning.jobs.retrieve(job_id)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-9tAjIOkWXaWAqEdrUr4JxRXF', created_at=1756524557, level='info', message='Fine-tuning job started', object='fine_tuning.job.event', data=None, type='message'),
 FineTuningJobEvent(id='ftevent-EVN6O7af8xOZh62NcttgAeXP', created_at=1756524554, level='info', message='Files validated, moving job to queued state', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-5cElhqfpOWGy4EfmFzqgPkOY', created_at=1756524448, level='info', message='Validating training file: file-FNyc5h3hrLjrE8GXt517P3 and validation file: file-YJLbNXNzvzLGz5BwoewzqZ', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-I08x06elVbO6PP0YSddhd1Qa', created_at=1756524448, level='info', message='Created fine-tuning job: ftjob-qJjQ4lSBSlt9yNVjMziJDZ3M', object='fine_tuning.job.event', data={}, type='message')]

In [11]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-biN1COfSRzY5heAwMp8yif7Q', created_at=1756524885, level='info', message='Step 154/200: training loss=0.41', object='fine_tuning.job.event', data={'step': 154, 'train_loss': 0.4146273136138916, 'total_steps': 200, 'train_mean_token_accuracy': 0.875}, type='metrics'),
 FineTuningJobEvent(id='ftevent-LmAda9qmI9gP5DIsJXw43mPS', created_at=1756524882, level='info', message='Step 153/200: training loss=0.50', object='fine_tuning.job.event', data={'step': 153, 'train_loss': 0.49959802627563477, 'total_steps': 200, 'train_mean_token_accuracy': 0.875}, type='metrics'),
 FineTuningJobEvent(id='ftevent-JJWcESLqxPPWC7erMlUbfwY4', created_at=1756524882, level='info', message='Step 152/200: training loss=0.52', object='fine_tuning.job.event', data={'step': 152, 'train_loss': 0.5162107944488525, 'total_steps': 200, 'train_mean_token_accuracy': 0.875}, type='metrics'),
 FineTuningJobEvent(id='ftevent-AzmVIFDBXapDy1hTx8PWidOj', created_at=1756524879, level='info', messag

In [12]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
def gpt_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name, 
        messages=messages_for(item),
        seed=42,
        max_tokens=7
    )
    reply = response.choices[0].message.content
    return get_price(reply)

In [17]:
print(gpt_fine_tuned(test[0]))

47.99


In [ ]:
Tester.test(gpt_fine_tuned, test)